In [0]:
df_bronze =  spark.read.table('workspace.`sports-pipeline`.football_matches_delta')
df_bronze.printSchema()
df_bronze.show(5)

In [0]:
from pyspark.sql.functions import col, to_timestamp, when, upper

df_silver = df_bronze.select(
    col('match_id'),
    to_timestamp(col('match_date'), "yyyy-MM-dd'T'HH:mm:ss'Z'").alias('match_date'),
    upper(col('status')).alias('status'),
    col('home_team'),
    col("away_team"),
    col('home_score').cast("integer"),
    col('away_score').cast("integer"),
    col("competition")
    ) \
    .withColumn("results",
        when(col("status") == "FINISHED",
            when(col("home_score") > col("away_score"), col("home_team"))
            .when(col("home_score") < col("away_score"), col("away_team"))
            .otherwise("Draw")
        )
        .when(col("status").isin("SCHEDULED", "TIMED"), "Not played yet")
        .when(col("status").isin("LIVE", "IN_PLAY"), "In progress")
        .when(col("status") == "PAUSED", "Half time")
        .when(col("status") == "POSTPONED", "Postponed")
        .when(col("status") == "SUSPENDED", "Suspended")
        .when(col("status") == "CANCELLED", "Cancelled")
        .otherwise("Unknown")
    ) \
    .dropDuplicates(["match_id"]) \
    .filter(col("match_id").isNotNull())

df_silver.show(10)

In [0]:
#Save Silver Table
df_silver.write\
    .format("delta")\
    .mode("overwrite")\
    .option("overwriteSchema", "true")\
    .saveAsTable("workspace.`sports-pipeline`.football_matches_silver")

print("Silver Table Saved")